# Esqueleto ARIMA para previsão de trajetória

Este notebook deixa uma função reutilizável para prever coordenadas futuras `x` e `y` a partir de frames observados. A função ajusta um ARIMA separado para cada coordenada e para cada grupo (por padrão, jogo, jogada e jogador).

In [ ]:
from typing import Sequence

import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
def forecast_arima_by_group(
    data: pd.DataFrame,
    steps: int,
    group_cols: Sequence[str] = ("gameId", "playId", "nflId"),
    time_col: str = "frameId",
    coordinate_cols: Sequence[str] = ("x", "y"),
    order: tuple[int, int, int] = (1, 1, 1),
) -> pd.DataFrame:
    """Prevê coordenadas futuras com ARIMA independente por grupo e coordenada.

    `data` deve conter observações históricas disponíveis até o instante
    da previsão. O resultado contém as chaves do grupo, o frame futuro
    e uma coluna de previsão para cada coordenada solicitada.
    """
    if steps < 1:
        raise ValueError("steps deve ser pelo menos 1")
    if not coordinate_cols:
        raise ValueError("Informe ao menos uma coluna de coordenada")

    required = [*group_cols, time_col, *coordinate_cols]
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise KeyError(f"Colunas ausentes: {missing}")

    predictions = []
    grouped = data.sort_values(time_col).groupby(
        list(group_cols), sort=False, dropna=False
    )

    for group_key, group in grouped:
        group = group.sort_values(time_col)
        key_values = group_key if isinstance(group_key, tuple) else (group_key,)
        group_values = dict(zip(group_cols, key_values))
        last_frame = int(group[time_col].iloc[-1])
        future_frames = range(last_frame + 1, last_frame + steps + 1)
        group_forecast = pd.DataFrame({time_col: list(future_frames)})

        for column in coordinate_cols:
            series = pd.to_numeric(group[column], errors="coerce").dropna()
            if len(series) < 3:
                raise ValueError(
                    f"Poucas observações válidas para {column} no grupo {group_values}"
                )
            fitted = ARIMA(series, order=order).fit()
            group_forecast[column] = fitted.forecast(steps=steps).to_numpy()

        for column, value in group_values.items():
            group_forecast[column] = value
        predictions.append(group_forecast)

    if not predictions:
        return pd.DataFrame(columns=[*group_cols, time_col, *coordinate_cols])
    return pd.concat(predictions, ignore_index=True)[
        [*group_cols, time_col, *coordinate_cols]
    ]

## Exemplo de chamada

Carregue os dados e aplique o recorte temporal exigido pela competição antes de chamar a função. As linhas abaixo ficam como modelo: substitua o caminho e ajuste o filtro de frames para o esquema real.

In [ ]:
# Exemplo (ajuste caminho, colunas e recorte temporal):
# tracking = pd.read_parquet("../data/processed/tracking.parquet")
# historico = tracking.loc[tracking["frameId"] <= frame_corte]
# previsoes = forecast_arima_by_group(
#     historico,
#     steps=numero_de_frames_futuros,
#     order=(1, 1, 1),
# )
# previsoes.head()